# Completing the Measurement: Compensation, Balance, and Protection
### *Modern Strain Gage Technology* — Companion Notebook

A transducer that has been designed, bonded, and wired is a sensing element with unknown systematic errors — not yet a measurement system. Before it can report a calibrated, traceable output, those errors have to be found and removed.

Companion notebook to Chapter 43 — Completing the Measurement: Compensation, Balance, and Protection.

Chapter 43 draws a careful line between three ideas that are easy to blur together. *Compensation* shapes the transducer so its output is stable and close to target — the job of the temperature-tracking copper and Balco resistors described in the chapter. *Adjustment* is the trimming that sets **zero** and **span** to their nominal numbers. *Calibration* is the separate act of *documenting* the finished relationship between applied load and output. This notebook works through the **adjustment** arithmetic: a representative transducer comes off the bench with a **3.2 mV** residual zero offset and a full-scale output that misses its target, and we apply zero balance and span correction in sequence to bring it to specification.

**This notebook demonstrates:**
1. The adjustment sequence — zero balance → span correction → adjusted full-scale verification — applied to a representative transducer, then confirmed across the full range.

In [ ]:
# Colab: install dependencies (uncomment on first run)
# !pip install numpy

import numpy as np

---
## 1 — The Adjustment Sequence: Zero Balance and Span Correction

Zero balance removes the residual bridge output that exists at zero load due to gage resistance mismatch, residual stresses in the spring element, and asymmetries in the bond and wiring. In this example, the raw zero output of 3.2 mV would appear as an apparent load offset if left uncorrected — every subsequent reading would be biased by that amount. Zeroing simply establishes a new reference level.

Span correction then scales the full-scale output to match the design target; equivalently, it sets the transducer's rated sensitivity (commonly quoted in mV/V of excitation). After zeroing, the full-scale output of 17.60 mV differs from the 20.00 mV target by 13.6%. The resulting span correction factor of 1.1364 is stored in the instrument or applied in post-processing to every subsequent reading. Because both steps are linear, a single offset-and-factor pair calibrates the entire range, not just the endpoints — we verify this below by sweeping across the full scale. Together, zero balance and span correction account for the dominant systematic errors in a newly built transducer; the remaining errors (linearity, hysteresis, creep) are characterized during the documented calibration that follows.

In [ ]:
# Representative bench readings for one transducer, in millivolts.
# Named constants keep the calibration inputs explicit and easy to change.
V_ZERO_RAW  = 3.2    # residual bridge output at zero load (initial imbalance)
V_FS_RAW    = 20.8   # output at rated full scale, before any adjustment
V_FS_TARGET = 20.0   # design-target full-scale output


def adjust_reading(v_raw, v_zero_raw, v_fs_raw, v_fs_target):
    """Apply zero balance then span correction to a raw bridge reading.

    Zero balance subtracts the residual no-load offset so the transducer
    reads zero at zero load. Span correction then multiplies the zeroed
    reading by a constant factor chosen so the rated full-scale reading
    lands exactly on the design target. Both corrections are linear, so a
    single (offset, factor) pair adjusts every reading in the range.

    Parameters
    ----------
    v_raw : float or np.ndarray
        Raw bridge output(s) to adjust, in mV.
    v_zero_raw, v_fs_raw, v_fs_target : float
        Raw zero output, raw full-scale output, and target full-scale
        output, all in mV.

    Returns
    -------
    float or np.ndarray
        Adjusted reading(s), in mV.
    """
    span_factor = v_fs_target / (v_fs_raw - v_zero_raw)
    return (v_raw - v_zero_raw) * span_factor


span_factor = V_FS_TARGET / (V_FS_RAW - V_ZERO_RAW)

print("TRANSDUCER ADJUSTMENT SEQUENCE")
print()
print(f"Raw zero output:     {V_ZERO_RAW:6.2f} mV")
print(f"Raw full-scale:      {V_FS_RAW:6.2f} mV")
print(f"After zero balance:  {V_FS_RAW - V_ZERO_RAW:6.2f} mV  (full-scale span)")
print(f"Span correction:     x {span_factor:.4f}")
print(f"Adjusted full-scale: "
      f"{adjust_reading(V_FS_RAW, V_ZERO_RAW, V_FS_RAW, V_FS_TARGET):6.2f} mV"
      f"  (target {V_FS_TARGET:.2f})")
print()

# The same two constants adjust every point in the range, not just the
# endpoints. Apply them to raw readings taken at 0, 25, 50, 75, 100% of the
# raw full-scale span and confirm the adjusted outputs step evenly to target.
raw_sweep = np.linspace(V_ZERO_RAW, V_FS_RAW, 5)
adj_sweep = adjust_reading(raw_sweep, V_ZERO_RAW, V_FS_RAW, V_FS_TARGET)

print("Raw (mV)  ->  Adjusted (mV)")
for v_raw, v_adj in zip(raw_sweep, adj_sweep):
    print(f"  {v_raw:6.2f}  ->  {v_adj:7.3f}")
print()
print("Next: document the load-vs-output relationship (calibration proper),")
print("then verify linearity, hysteresis, and repeatability.")

---
## Where to take this next

The adjustment arithmetic above is deliberately spare — it is a starting point, not the whole compensation network. A few concrete extensions turn it into a fuller model of Chapter 43:

- **Add the temperature corrections.** Implement Eq. 43.1 and Eq. 43.3 to size the copper zero-shift resistor and the Balco span-shift resistor for a chosen bridge, then plot the residual zero and span error versus temperature before and after compensation.
- **Model the trimming ladders.** Represent each foil ladder as a discrete set of rungs with a fixed resistance increment (roughly 0.005 Ω per rung for the copper example), and write a routine that counts the rungs to cut to reach a target correction — reproducing the technician's scalpel step, quantization and all.
- **Fold in the leftover errors.** Layer synthetic nonlinearity, hysteresis, and creep onto the raw sweep (seed any randomness with `np.random.default_rng(0)` for reproducibility) and compute the resulting nonlinearity and hysteresis figures — the numbers that ultimately land on the calibration certificate.